In [2]:
#EXPLANATION OF REGRESSION RESULTS 

# MULTICOLLINEARITY CHECK:
# - RAD (Highway Accessibility, VIF = 7.44) was removed — VIF > 5 (class rule)
# - TAX (Property Tax Rate, VIF = 8.20) was removed — VIF > 5 (class rule)
# - Both were removed because they correlate too strongly with each other.
#   This makes sense: areas with better highway access tend to also have
#   higher property tax rates, so they carry overlapping information.
# - All remaining variables had VIF scores below 5, meaning no further
#   multicollinearity issues were detected.

# OVERALL MODEL QUALITY:
# - The final model has an R² of 0.880, meaning it explains 88% of the
#   variation in house prices across neighbourhoods. This is a very strong model.
# - Removing RAD and TAX barely changed anything (was 88.3%) — proof that
#   those variables were not adding unique information anyway.
# - The overall model is statistically significant (p < .001), meaning these
#   results are very unlikely to be coincidence.

# WHAT LOWERS HOUSE PRICES (negative coefficients, all p < .05):
# - Higher crime rate (-$3,893 per unit increase) — unsafe areas are worth less
# - More air pollution (-$213,839 per unit of NOX) — the strongest negative effect
# - More old houses in the area (-$792 per 1% increase in pre-1940 homes)
# - Farther from employment centres (-$8,993 per unit of distance)
# - Higher pupil-teacher ratio (-$6,413 per extra

In [4]:
import pandas as pd
import statsmodels.formula.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = pd.read_csv("1632349boston.csv")

print("=" * 70)
print("ASSIGNMENT 2: MULTIPLE REGRESSION - MEDIAN HOUSE PRICE (MEDV)")
print("=" * 70)

#Step 1: Create dummy variables 
# I manually drop the correct reference categories:
#   CHAS     → reference = 'no'           → drop CHAS_no
#   LANDMARK → reference = 'No Landmark'  → drop LANDMARK_No_Landmark

df_model = pd.get_dummies(df, columns=['CHAS', 'LANDMARK'], dtype=int)
df_model.columns = df_model.columns.str.replace(' ', '_')
df_model = df_model.drop(columns=['CHAS_no', 'LANDMARK_No_Landmark'])

print("\nColumns after dummy encoding:")
print(df_model.columns.tolist())

#Step 2: Run the full regression model 
# sm.ols('Dependent ~ Independent1 + Independent2 + ...', data = df).fit()

formula_full = ('MEDV ~ CRIM + INDUS + NOX + RM + AGE + DIS + RAD + TAX + PTRATIO '
                '+ CHAS_yes '
                '+ LANDMARK_Museum + LANDMARK_Park + LANDMARK_Shopping_Mall + LANDMARK_Stadium')

model_full = sm.ols(formula_full, data=df_model).fit()

print("\n" + "=" * 70)
print("FULL MODEL SUMMARY (before multicollinearity check)")
print("=" * 70)
print(model_full.summary())

#Step 3: Check multicollinearity using VIF

print("\n" + "=" * 70)
print("MULTICOLLINEARITY CHECK: Variance Inflation Factors (VIF)")
print("=" * 70)
print("Rule from class: VIF > 5 = multicollinearity problem → remove the variable\n")

X_vars = ['CRIM', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO',
          'CHAS_yes',
          'LANDMARK_Museum', 'LANDMARK_Park', 'LANDMARK_Shopping_Mall', 'LANDMARK_Stadium']

X = df_model[X_vars].copy().astype(float)
X.insert(0, 'Intercept', 1.0)

vif_data = pd.DataFrame()
vif_data['Variable'] = X_vars
vif_data['VIF']      = [variance_inflation_factor(X.values, i + 1)
                        for i in range(len(X_vars))]
vif_data['Problem?'] = vif_data['VIF'].apply(
    lambda v: 'REMOVE (VIF > 5)' if v > 5 else 'OK')

print(vif_data.to_string(index=False))

high_vif = vif_data[vif_data['VIF'] > 5]['Variable'].tolist()
print(f"\nVariables with VIF > 5 (removed from final model): {high_vif if high_vif else 'None'}")

#Step 4: Run final model after removing high-VIF variables
final_vars    = [v for v in X_vars if v not in high_vif]
formula_final = 'MEDV ~ ' + ' + '.join(final_vars)

print("\n" + "=" * 70)
print("FINAL MODEL (after removing multicollinear variables)")
print(f"Formula used: {formula_final}")
print("=" * 70)

model_final = sm.ols(formula_final, data=df_model).fit()
print(model_final.summary())

#Step 5: APA formatted table 
print("\n" + "=" * 100)
print("APA TABLE: Multiple Regression Results Predicting Median Home Value (MEDV, in $1,000s)")
print("=" * 100)

params   = model_final.params
std_err  = model_final.bse
t_vals   = model_final.tvalues
p_vals   = model_final.pvalues
conf_int = model_final.conf_int()

labels = {
    'Intercept':              'Intercept',
    'CRIM':                   'Crime Rate',
    'INDUS':                  'Non-Retail Business (%)',
    'NOX':                    'Nitric Oxides Concentration',
    'RM':                     'Avg. Rooms per House',
    'AGE':                    '% Homes Built Before 1940',
    'DIS':                    'Distance to Employment Centre',
    'RAD':                    'Highway Accessibility',
    'TAX':                    'Property Tax Rate',
    'PTRATIO':                'Pupil-Teacher Ratio',
    'CHAS_yes':               'Borders Charles River (yes)',
    'LANDMARK_Museum':        'Landmark: Museum',
    'LANDMARK_Park':          'Landmark: Park',
    'LANDMARK_Shopping_Mall': 'Landmark: Shopping Mall',
    'LANDMARK_Stadium':       'Landmark: Stadium',
}

print(f"\n{'Variable':<32} {'B':>9} {'SE':>9} {'t':>9} {'p':>9} {'95% CI Lower':>14} {'95% CI Upper':>14}")
print("-" * 100)

for var in params.index:
    label = labels.get(var, var)
    b     = params[var]
    se    = std_err[var]
    t     = t_vals[var]
    p     = p_vals[var]
    ci_lo = conf_int.loc[var, 0]
    ci_hi = conf_int.loc[var, 1]

    if p < 0.001:
        sig = '***'
    elif p < 0.01:
        sig = '**'
    elif p < 0.05:
        sig = '*'
    else:
        sig = ''

    p_str = '< .001' if p < 0.001 else f'{p:.3f}'
    print(f"{label:<32} {b:>9.3f} {se:>9.3f} {t:>9.3f} {p_str:>9} {ci_lo:>14.3f} {ci_hi:>14.3f}  {sig}")

print("-" * 100)
print(f"\nNote. N = {int(model_final.nobs)}. "
      f"R² = {model_final.rsquared:.3f}, "
      f"Adjusted R² = {model_final.rsquared_adj:.3f}, "
      f"F({int(model_final.df_model)}, {int(model_final.df_resid)}) = {model_final.fvalue:.3f}, "
      f"p {'< .001' if model_final.f_pvalue < 0.001 else f'= {model_final.f_pvalue:.3f}'}.")
print("* p < .05. ** p < .01. *** p < .001.")
print("\nReference categories: CHAS = 'no'; LANDMARK = 'No Landmark'.")
print(f"Variables removed due to multicollinearity (VIF > 5): {high_vif if high_vif else 'None'}")

# PART 2: STANDARDIZATION — Which has a bigger influence: CRIM or RM?
# You cannot compare coefficients of CRIM and RM directly because they are
# measured on completely different scales. The solution is standardization:
# we rescale all continuous variables to the same scale (mean=0, sd=1)
# so their coefficients become directly comparable.
# NOT dummy variables (CHAS, LANDMARK) and NOT the dependent variable (MEDV).

print("\n" + "=" * 70)
print("PART 2: STANDARDIZATION")
print("Which has a bigger influence on house price: Crime Rate or Avg. Rooms?")
print("=" * 70)
print("\nReason: CRIM and RM are on different scales so we cannot compare")
print("their coefficients directly. Standardization fixes this.")


continuous_vars = [v for v in final_vars
                   if v not in ['CHAS_yes', 'LANDMARK_Museum', 'LANDMARK_Park',
                                'LANDMARK_Shopping_Mall', 'LANDMARK_Stadium']]

print(f"\nVariables being standardized: {continuous_vars}")
print("Variables NOT standardized (dummies): CHAS_yes, LANDMARK_*")

df_scaled = df_model.copy()
for col in continuous_vars:
    col_mean = df_model[col].mean()
    col_std = df_model[col].std(ddof=0)
    if col_std == 0:
        df_scaled[col] = 0.0
    else:
        df_scaled[col] = (df_model[col] - col_mean) / col_std


model_standardized = sm.ols(formula_final, data=df_scaled).fit()

print("\n" + "=" * 70)
print("STANDARDIZED MODEL SUMMARY")
print("=" * 70)
print(model_standardized.summary())

#Compare CRIM and RM standardized coefficients 
coef_crim = model_standardized.params['CRIM']
coef_rm   = model_standardized.params['RM']

print("\n" + "=" * 70)
print("COMPARISON: CRIM vs RM (standardized coefficients)")
print("=" * 70)
print(f"Crime Rate (CRIM) standardized coefficient : {coef_crim:.3f}")
print(f"Avg. Rooms (RM)   standardized coefficient : {coef_rm:.3f}")
print(f"\nAbsolute values for comparison:")
print(f"  |CRIM| = {abs(coef_crim):.3f}")
print(f"  |RM|   = {abs(coef_rm):.3f}")

if abs(coef_rm) > abs(coef_crim):
    winner = "Avg. Rooms per House (RM)"
    loser  = "Crime Rate (CRIM)"
else:
    winner = "Crime Rate (CRIM)"
    loser  = "Avg. Rooms per House (RM)"

print(f"\n→ {winner} has a bigger influence on median house price than {loser}.")
params_s   = model_standardized.params
#APA table for standardized model 
print("\n" + "=" * 100)
print("APA TABLE: Standardized Regression Results Predicting Median Home Value (MEDV)")
print("=" * 100)

params_s   = model_standardized.params
std_err_s  = model_standardized.bse
t_vals_s   = model_standardized.tvalues
p_vals_s   = model_standardized.pvalues
conf_int_s = model_standardized.conf_int()

print(f"\n{'Variable':<32} {'β (std)':>9} {'SE':>9} {'t':>9} {'p':>9} {'95% CI Lower':>14} {'95% CI Upper':>14}")
print("-" * 100)

for var in params_s.index:
    label = labels.get(var, var)
    b     = params_s[var]
    se    = std_err_s[var]
    t     = t_vals_s[var]
    p     = p_vals_s[var]
    ci_lo = conf_int_s.loc[var, 0]
    ci_hi = conf_int_s.loc[var, 1]

    if p < 0.001:
        sig = '***'
    elif p < 0.01:
        sig = '**'
    elif p < 0.05:
        sig = '*'
    else:
        sig = ''

    p_str = '< .001' if p < 0.001 else f'{p:.3f}'
    print(f"{label:<32} {b:>9.3f} {se:>9.3f} {t:>9.3f} {p_str:>9} {ci_lo:>14.3f} {ci_hi:>14.3f}  {sig}")

print("-" * 100)
print(f"\nNote. N = {int(model_standardized.nobs)}. "
      f"R² = {model_standardized.rsquared:.3f}, "
      f"Adjusted R² = {model_standardized.rsquared_adj:.3f}, "
      f"F({int(model_standardized.df_model)}, {int(model_standardized.df_resid)}) = {model_standardized.fvalue:.3f}, "
      f"p {'< .001' if model_standardized.f_pvalue < 0.001 else f'= {model_standardized.f_pvalue:.3f}'}.")
print("* p < .05. ** p < .01. *** p < .001.")

print("β = standardized coefficient. Only continuous variables were standardized.")
print("Reference categories: CHAS = 'no'; LANDMARK = 'No Landmark'.")

ASSIGNMENT 2: MULTIPLE REGRESSION - MEDIAN HOUSE PRICE (MEDV)

Columns after dummy encoding:
['MEDV', 'CRIM', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'CHAS_yes', 'LANDMARK_Museum', 'LANDMARK_Park', 'LANDMARK_Shopping_Mall', 'LANDMARK_Stadium']

FULL MODEL SUMMARY (before multicollinearity check)
                            OLS Regression Results                            
Dep. Variable:                   MEDV   R-squared:                       0.883
Model:                            OLS   Adj. R-squared:                  0.879
Method:                 Least Squares   F-statistic:                     237.5
Date:                Fri, 27 Mar 2026   Prob (F-statistic):          6.21e-195
Time:                        09:19:14   Log-Likelihood:                -2523.8
No. Observations:                 455   AIC:                             5078.
Df Residuals:                     440   BIC:                             5139.
Df Model:                          14             

In [3]:
#EXPLANATION OF STANDARDIZATION RESULTS

# WHY WE USED STANDARDIZATION:
# - Before standardization, CRIM had a coefficient of -3.89 and RM had 27.47.
# - You cannot compare these directly because they are on different scales.
#   A "1 unit increase" in CRIM means something completely different from
#   a "1 unit increase" in RM — so comparing them would be unfair.
# - Standardization puts both variables on the same scale (mean = 0, sd = 1),
#   so we can now fairly compare which one moves house prices more.
# - Note: we only standardized the continuous variables (CRIM, INDUS, NOX,
#   RM, AGE, DIS, PTRATIO). Dummy variables like CHAS and LANDMARK were
#   NOT standardized, as this does not make sense for categorical variables.

# WHAT THE STANDARDIZED COEFFICIENTS MEAN:
# - CRIM (standardized β = -30.71, p < .001):
#   When crime rate goes up by 1 standard deviation, the median house price
#   drops by $30,711 on average, controlling for all other variables.
# - RM (standardized β = 19.03, p < .001):
#   When the average number of rooms goes up by 1 standard deviation, the
#   median house price increases by $19,027 on average, controlling for
#   all other variables.

# CONCLUSION — WHICH HAS A BIGGER INFLUENCE?
# - |CRIM| = 30.71  vs  |RM| = 19.03
# - Crime Rate (CRIM) has a bigger influence on median house price than
#   the average number of rooms (RM).
# - In other words: living in a high-crime area hurts your house price
#   more than having an extra room helps it.

# HOW TO EXPLAIN THIS TO A FRIEND (non-technical summary):
# - "We wanted to know which matters more for house prices: crime or room size."
# - "The problem is that crime rate and number of rooms are measured in totally
#    different units — so we first had to put them on the same scale."
# - "Once we did that, we could see that crime rate has a much bigger impact
#    than the number of rooms — a big jump in crime hurts prices by about
#    $30,711, while a big jump in room size only adds about $19,027."
# - "So if your friend is looking to buy a house, living in a safe neighbourhood
#    matters even more than having a big house!"